# Praktika: Naive Bayes

## 0. Ariketa: Aurre-ebaluazioa
Hurrengo [datu-bilduma](https://drive.google.com/file/d/1cX5QnIu3RnfEyM66aatOJDYvhx7xRf66/view?usp=sharing) izanda. Hurrengo galderak erantzun.
1. Zein da Spam izatearen probabilitatea?
2. Zein da Ham izatearen probabilitatea?

In [1]:
## importak

import pandas as pd
from sklearn.naive_bayes import MultinomialNB
#from gensim.models.word2vec import Word2Vec
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [83]:
## datu bilduma kargatu
df = pd.read_csv('spamhamdata.csv', sep='\t', quoting=3, header=None, names=['Klaseak', 'Datuak'])
df.tail()

,Klaseak,Datuak
5569,spam,This is the 2nd time we have tried 2 contact u...
5570,ham,Will ü b going to esplanade fr home?
5571,ham,"Pity, * was in mood for that. So...any other s..."
5572,ham,The guy did some bitching but I acted like i'd...
5573,ham,Rofl. Its true to its name


In [84]:
## probabilitateak atera

kont = df['Klaseak'].value_counts()
prob_spam = kont['spam'] / len(df)*100
prob_ham = kont['ham'] / len(df)*100

print(f'Probabilitatea spam: %{prob_spam}')
print(f'Probabilitatea ham: %{prob_ham}')

Probabilitatea spam: %13.40150699677072
Probabilitatea ham: %86.59849300322928


## 1. Ariketa: Testu prozesamendua
Word2Vec erabiliz, hurrengo galderak erantzun.
1. Zenbat da 'winner' eta 'mobile' hitzen arteko erlazioa?
2. Zein dira 'coffee' hitzarekin lotura gehien daukaten 5 hitzak?

### Testu zarata kendu

In [85]:
## testu garbiketa

datuak = df['Datuak'].tolist()

print(df['Datuak'][13])

def garbitu_testua(testua):

    # minuskulak bihurtu
    testua = testua.lower()

    # url kendu
    testua = re.sub(r'http\S+|www\S+|https\S+', '', testua, flags=re.MULTILINE)

    # kendu zenbakiak duten hitzak
    testua = re.sub(r"\w*\d\w*", "", testua)

    # karaktere bereziak kendu
    testua = re.sub(r"[^'a-z\s]", "", testua)

    # kendu gehiegizko hutsuneak
    testua = re.sub(r"\s+", " ", testua).strip()
    
    return testua

# testua paragrafoka
datuak = [garbitu_testua(datuak[index]) for index in range(len(datuak))]
print(datuak[13])

I've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.
i've been searching for the right words to thank you for this breather i promise i wont take your help for granted and will fulfil my promise you have been wonderful and a blessing at all times


In [86]:
print(datuak[13])

# Decontract English contractions
contractions = {
    "ain\'t": "are not", "aren\'t": "are not", "can\'t": "can not", "can\'t\'ve": "can not have",
    "could\'ve": "could have", "couldn\'t": "could not", "couldn\'t\'ve": "could not have",
    "didn\'t": "did not", "doesn\'t": "does not", "don\'t": "do not", "hadn\'t": "had not",
    "hadn\'t\'ve": "had not have", "hasn\'t": "has not", "haven\'t": "have not",
    "he\'d": "he would", "he\'d\'ve": "he would have", "he\'ll": "he will", "he\'s": "he is",
    "how\'d": "how did", "how\'ll": "how will", "how\'s": "how is", "i\'d": "i would",
    "i\'d\'ve": "i would have", "i\'ll": "i will", "i\'m": "i am", "i\'ve": "i have",
    "isn\'t": "is not", "it\'d": "it would", "it\'d\'ve": "it would have", "it\'ll": "it will",
    "it\'s": "it is", "let\'s": "let us", "ma\'am": "madam", "might\'ve": "might have",
    "mightn\'t": "might not", "must\'ve": "must have", "mustn\'t": "must not",
    "needn\'t": "need not", "shan\'t": "shall not", "she\'d": "she would",
    "she\'d\'ve": "she would have", "she\'ll": "she will", "she\'s": "she is",
    "should\'ve": "should have", "shouldn\'t": "should not", "that\'d": "that would",
    "that\'s": "that is", "there\'d": "there would", "there\'s": "there is",
    "they\'d": "they would", "they\'ll": "they will", "they\'re": "they are",
    "they\'ve": "they have", "wasn\'t": "was not", "we\'d": "we would",
    "we\'ll": "we will", "we\'re": "we are", "we\'ve": "we have", "weren\'t": "were not",
    "what\'ll": "what will", "what\'re": "what are", "what\'s": "what is",
    "what\'ve": "what have", "when\'s": "when is", "where\'d": "where did",
    "where\'s": "where is", "who\'ll": "who will", "who\'s": "who is",
    "won\'t": "will not", "wouldn\'t": "would not", "you\'d": "you would",
    "you\'ll": "you will", "you\'re": "you are", "you\'ve": "you have"
}

# Replace contractions
for index, p in enumerate(datuak):
    for contraction, expansion in contractions.items():
        p = p.replace(contraction, expansion)
    datuak[index] = p

print(datuak[13])

i've been searching for the right words to thank you for this breather i promise i wont take your help for granted and will fulfil my promise you have been wonderful and a blessing at all times
i have been searching for the right words to thank you for this breather i promise i wont take your help for granted and will fulfil my promise you have been wonderful and a blessing at all times


### Tokenizazioa

In [92]:
# nltk kargatu
nltk.download('stopwords')
nltk.download('punkt')  # Required for word_tokenize
nltk.download('punkt_tab')  # Additional tokenizer data

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return tokens


# testua tokenizatu
filtered_tokens = [preprocess_text(p) for p in datuak]

len(filtered_tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ikasle\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ikasle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ikasle\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


5574

In [88]:
## winner mobile



In [89]:
## coffee




## 2. Ariketa: Modelo sorrera eta ebaluazioa

Naive Bayes erabiliz, iragarpen modelo bat sortu.

Hurrengo galdera erantzun.

> Zein Naive Bayes erabiliko duzu. Zergatik?

In [90]:
## modeloa kargatu


In [91]:
## modelo ebaluazioa

